## ENV Setup

In [25]:
import pandas as pd

In [11]:
path_folder = "Datasets/"
path_claims = "ClaimDetails_for_distribution.xlsx"
path_adjuster = "CatRosterReportDayOf_for_distribution_cleaned.xlsx"
path_dictionary = "Data_Set_decode_table_updated.xlsx"

## Datasets & Dictionary

In [12]:
ds_dictionary_claims = pd.read_excel(
    path_folder+path_dictionary,
    sheet_name=0
)

ds_dictionary_adjuster = pd.read_excel(
    path_folder+path_dictionary,
    sheet_name=1
)

ds_claims_main = pd.read_excel(
    path_folder+path_claims,
    sheet_name=0
)

ds_adjuster_main = pd.read_excel(
    path_folder+path_adjuster,
    sheet_name=0
)

In [13]:
dict_claims_var_col = ds_dictionary_claims.columns[0]
dict_adjuster_var_col = ds_dictionary_adjuster.columns[0]

ds_dictionary_claims["exists_in_main"] = (
    ds_dictionary_claims[dict_claims_var_col]
        .astype(str)
        .isin(ds_claims_main.columns)
)

ds_dictionary_adjuster["exists_in_main"] = (
    ds_dictionary_adjuster[dict_adjuster_var_col]
        .astype(str)
        .isin(ds_adjuster_main.columns)
)

In [39]:
len(ds_claims_main)

1065

In [46]:
# ============================================================
# AUDIT ACCIDENT ZIP IN ORIGINAL DATASET
# ============================================================

import pandas as pd
import numpy as np
import re

zip_audit = ds_claims_main.copy()

# Standardize claim ID if present
if "Claim Number" in zip_audit.columns:
    zip_audit["Claim Number"] = zip_audit["Claim Number"].astype(str)

# Keep raw copy
zip_audit["Accident Zip_raw"] = zip_audit["Accident Zip"]

# Convert to string carefully
zip_audit["Accident Zip_str"] = zip_audit["Accident Zip_raw"].astype(str).str.strip()

# Common missing-like forms
missing_like = ["", "nan", "none", "null", "nat"]

zip_audit["zip_is_na"] = zip_audit["Accident Zip_raw"].isna()
zip_audit["zip_is_blankish"] = zip_audit["Accident Zip_str"].str.lower().isin(missing_like)

# Clean numeric-looking zips like 30043.0 -> 30043
zip_audit["zip_digits_only"] = (
    zip_audit["Accident Zip_str"]
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"[^0-9]", "", regex=True)
)

# Valid ZIP rules:
# 5 digits or 9 digits are accepted
zip_audit["zip_len"] = zip_audit["zip_digits_only"].str.len()
zip_audit["zip_valid_format"] = zip_audit["zip_len"].isin([5, 9])

# Broad missing/problem flag
zip_audit["zip_problem"] = (
    zip_audit["zip_is_na"] |
    zip_audit["zip_is_blankish"] |
    (~zip_audit["zip_valid_format"])
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
zip_summary = pd.DataFrame({
    "metric": [
        "rows_total",
        "zip_is_na",
        "zip_is_blankish",
        "zip_invalid_format",
        "zip_problem_total"
    ],
    "value": [
        len(zip_audit),
        int(zip_audit["zip_is_na"].sum()),
        int(zip_audit["zip_is_blankish"].sum()),
        int((~zip_audit["zip_valid_format"]).sum()),
        int(zip_audit["zip_problem"].sum())
    ]
})

print("\n================ ACCIDENT ZIP SUMMARY ================\n")
display(zip_summary)

# ------------------------------------------------------------
# Show problematic rows
# ------------------------------------------------------------
zip_problem_rows = zip_audit.loc[zip_audit["zip_problem"]].copy()

cols_to_show = [
    c for c in [
        "Claim Number",
        "Accident Zip_raw",
        "Accident Zip_str",
        "zip_digits_only",
        "zip_len",
        "zip_is_na",
        "zip_is_blankish",
        "zip_valid_format",
        "zip_problem",
        "On-Site Handling",
        "Virtual Handling",
        "Territory",
        "Accident State",
        "Accident City"
    ] if c in zip_problem_rows.columns
]

print("\n================ PROBLEMATIC ACCIDENT ZIP ROWS ================\n")
display(zip_problem_rows[cols_to_show].head(100))

# ------------------------------------------------------------
# Onsite-only summary
# ------------------------------------------------------------
if "On-Site Handling" in zip_audit.columns:
    onsite_mask = zip_audit["On-Site Handling"].fillna(False).astype(bool)

    onsite_zip_summary = pd.DataFrame({
        "metric": [
            "onsite_rows_total",
            "onsite_zip_problem_total",
            "onsite_zip_is_na",
            "onsite_zip_is_blankish",
            "onsite_zip_invalid_format"
        ],
        "value": [
            int(onsite_mask.sum()),
            int(zip_audit.loc[onsite_mask, "zip_problem"].sum()),
            int(zip_audit.loc[onsite_mask, "zip_is_na"].sum()),
            int(zip_audit.loc[onsite_mask, "zip_is_blankish"].sum()),
            int((~zip_audit.loc[onsite_mask, "zip_valid_format"]).sum())
        ]
    })

    print("\n================ ONSITE ACCIDENT ZIP SUMMARY ================\n")
    display(onsite_zip_summary)

    print("\n================ ONSITE ROWS WITH ZIP PROBLEMS ================\n")
    display(zip_audit.loc[onsite_mask & zip_audit["zip_problem"], cols_to_show].head(100))


================ ACCIDENT ZIP SUMMARY ================



,metric,value
0,rows_total,1065
1,zip_is_na,0
2,zip_is_blankish,3
3,zip_invalid_format,3
4,zip_problem_total,3



================ PROBLEMATIC ACCIDENT ZIP ROWS ================



,Claim Number,Accident Zip_raw,Accident Zip_str,zip_digits_only,zip_len,zip_is_na,zip_is_blankish,zip_valid_format,zip_problem,Territory,Accident State,Accident City
144,10318529,,,,0,False,True,False,True,NaN,Alabama,TUSCALOOSA
767,40686860,,,,0,False,True,False,True,NaN,Tennessee,HENDERSONVILL
1061,76778644,,,,0,False,True,False,True,NaN,Tennessee,MADISON


In [14]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [38]:
ds_dictionary_claims[ds_dictionary_claims["exists_in_main"] == True]

,Claim Details Tab,Data Definition,Sample 1,Sample 2,Notes on potential usage,exists_in_main
0,Weather,Pre-applied Data Filter for Weather Cause of Loss,Weather,Weather,NaN,True
5,Territory,pre-defined Claim Office Sub-territory,Alabama - Middle,Alabama - Middle,NaN,True
6,Claim Number,Primary Key for Claim Data sets,GTE9304,I4P2271,NaN,True
7,CAT Code,Property Claim Services - Industry Cat Code,82,82,NaN,True
8,CAT Severity Code,Claim Severity/Complexity based on Customer Description of Loss at First Notice of Loss. listed as 1-5),3,5,Potentially important!,True
9,Loss Date,Date claim occurred,2023-12-09 00:00:00,2023-12-10 00:00:00,Few dates,True
10,NOL Date,Date the Claim was reported / notice of loss (NOL),2023-12-10 00:00:00,2023-12-10 00:00:00,NaN,True
24,Peril Description,"discrete list of claims damage types (Wind, water, hail, fire, etc.",Wind,Wind,Potentially important!,True
25,Cause Of Injury Text,free form description of the damage,TORNADO CAME THROUGH AND TREE WENT THROUGH INSURED'S ROOF PUNCTURING T,"HEAVY STORMS WITH WIND, HAIL, AND TREES DOWN. EXTENSIVE DAMAGE THROUGH",extract common terms from titles in addition to individual events,True
26,Peril Group,"discrete list of claims damage types (Wind, water, hail, fire, etc.) (selectively grouped)",Wind,Wind,Potentially important!,True


In [35]:
ds_dictionary_adjuster[ds_dictionary_adjuster["exists_in_main"] == True]

,Cat Roster Report Tab,Data Category,Data Definition,Sample 1,Sample 2,Notes on potential usage,exists_in_main
0,TIES Id,Core Job,System ID (used across platforms),N822D5,N8762C,NaN,True
3,Org Group,Core Job,Reports-to office based on HR Org Chart to 1st Full VP in system),Upper Midwest,Upper Midwest,NaN,True
4,Department,Core Job,based on Cost Center Code,Upper Midwest PI Prop,Upper Midwest PI Prop,extract common terms from titles,True
5,Location,Core Job,based on HR location code,St. Paul,West Des Moines-Jordan Creek,NaN,True
6,HR Job Title,Core Job,From HR,"Claim Rep Trainee, Outside Property","Claim Rep Trainee, Outside Property",extract common terms from titles,True
8,ERT Champion Ind,Cat Job,Pre-identifier as a Cat Response ER (Enterprise response team = cross trained team of non-property claim handlers),,N,NaN,True
9,ERT Champion,Cat Job,what ERT Champion assigned to,NaN,NaN,NaN,True
10,WFM Tour Supervisor Name,Cat Job,supervisor while deployed,NaN,NaN,NaN,True
11,Preferred Role,Cat Job,"discrete list of 'Cat Jobs"" including ""Claim Handler""",Property Outside Claim Rep,NaN,NaN,True
12,Availability,Cat Job,discrete list of current status,NaN,NaN,NaN,True


In [8]:
pd.reset_option("display.max_rows")
pd.reset_option("display.max_columns")
pd.reset_option("display.width")
pd.reset_option("display.max_colwidth")